# Terrain features

This notebook derives municipality-level terrain indicators relevant to
bicycle and e-bike adoption.

The objective is to characterize not only elevation, but also terrain
variability and slope, which may influence the relative attractiveness
of electric assistance compared with a conventional bicycle.

In [ ]:
# ============================================================================
# Imports
# ============================================================================

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from hermes.config import (
    RAW_DIR,
    PREPARED_DIR,
    FIGURES_DIR, 
    RGE_ALTI_MNT_DIR,
    CASE_STUDY_DEM_PATH,
    CASE_STUDY_SLOPE_PATH,
)
from hermes.utils import (
    download_dataset,
    get_dataset
)

from hermes.elevation import (
    build_mnt_index,
    build_tile_grid,
    compare_municipality_slope_resolutions,
    compare_slope_resolutions,
    compute_slope,
    find_tile_for_point,
    get_tile_coordinates,
    prepare_case_study_mnt,
    resample_elevation,
    slope_statistics,
    build_case_study_dem,
    build_slope_raster,
    compute_municipality_terrain_features,
)

from rasterio.features import shapes
from shapely.geometry import shape

In [ ]:
# ============================================================================
# Load prepared municipality data
# ============================================================================

municipality = pd.read_parquet(
    PREPARED_DIR / "municipality.parquet"
)

municipality.head()

In [ ]:
# ============================================================================
# Load municipality boundaries
# ============================================================================

municipality_boundaries = gpd.read_parquet(
    PREPARED_DIR / "municipality_boundaries.parquet"
)

municipality_boundaries.head()

In [ ]:
# ============================================================================
# Inspect municipality boundaries
# ============================================================================

print(f"Municipalities: {len(municipality_boundaries):,}")
print(f"CRS: {municipality_boundaries.crs}")

print("\nGeometry types:")
print(municipality_boundaries.geometry.geom_type.value_counts())

print("\nColumns:")
print(municipality_boundaries.columns.tolist())

print("\nBounds:")
print(municipality_boundaries.total_bounds)

In [ ]:
# ============================================================================
# Validate municipality / boundary matching
# ============================================================================

municipality_codes = set(
    municipality["insee_code"]
    .astype("string")
)

boundary_codes = set(
    municipality_boundaries["insee_code"]
    .astype("string")
)

print(
    f"Municipality table codes: {len(municipality_codes):,}"
)

print(
    f"Boundary codes: {len(boundary_codes):,}"
)

print(
    f"Missing boundaries: "
    f"{len(municipality_codes - boundary_codes):,}"
)

print(
    f"Boundaries without municipality data: "
    f"{len(boundary_codes - municipality_codes):,}"
)

In [ ]:
# ============================================================================
# Build municipality geodataframe
# ============================================================================

municipality_attributes = municipality.drop(
    columns=["geometry"],
    errors="ignore",
)

municipality_geo = municipality_attributes.merge(
    municipality_boundaries[
        [
            "insee_code",
            "geometry",
        ]
    ],
    on="insee_code",
    how="left",
)

municipality_geo = gpd.GeoDataFrame(
    municipality_geo,
    geometry="geometry",
    crs=municipality_boundaries.crs,
)

print(f"Rows: {len(municipality_geo):,}")

print(
    f"Missing geometries: "
    f"{municipality_geo.geometry.isna().sum():,}"
)

municipality_geo.head()

In [ ]:
# ============================================================================
# Inspect available elevation data
# ============================================================================

elevation_columns = [
    "mean_elevation",
    "min_elevation",
    "max_elevation",
]

municipality_geo[
    elevation_columns
].describe()

In [ ]:
# ============================================================================
# Diagnose inconsistent elevation values
# ============================================================================

invalid_elevation = municipality_geo[
    (
        municipality_geo["min_elevation"]
        > municipality_geo["max_elevation"]
    )
    | (
        municipality_geo["mean_elevation"]
        < municipality_geo["min_elevation"]
    )
    | (
        municipality_geo["mean_elevation"]
        > municipality_geo["max_elevation"]
    )
].copy()

invalid_elevation["elevation_range"] = (
    invalid_elevation["max_elevation"]
    - invalid_elevation["min_elevation"]
)

print(
    f"Municipalities with inconsistent elevation values: "
    f"{len(invalid_elevation):,}"
)

invalid_elevation[
    [
        "insee_code",
        "municipality",
        "department_code",
        "mean_elevation",
        "min_elevation",
        "max_elevation",
        "elevation_range",
    ]
].sort_values(
    "elevation_range"
)

## Terrain data quality assessment

The elevation attributes available in the municipality reference dataset
contain substantial inconsistencies. In particular, 1,374 municipalities
have elevation values for which the reported mean does not lie between
the minimum and maximum, or the minimum exceeds the maximum.

Rather than applying ad-hoc corrections, elevation statistics will therefore
be recomputed directly from a common digital elevation model (DEM). This
also ensures consistency between elevation and slope-derived terrain features.

In [ ]:
# ============================================================================
# Define HERMES case-study municipalities
# ============================================================================

villefranche_agglo = [
    ("Arnas", "69"),
    ("Blacé", "69"),
    ("Cogny", "69"),
    ("Denicé", "69"),
    ("Gleizé", "69"),
    ("Jassans-Riottier", "01"),
    ("Lacenas", "69"),
    ("Le Perréon", "69"),
    ("Limas", "69"),
    ("Montmelas-Saint-Sorlin", "69"),
    ("Rivolet", "69"),
    ("Salles-Arbuissonnas-en-Beaujolais", "69"),
    ("Saint-Cyr-le-Chatoux", "69"),
    ("Saint-Étienne-des-Oullières", "69"),
    ("Saint-Julien", "69"),
    ("Vaux-en-Beaujolais", "69"),
    ("Villefranche-sur-Saône", "69"),
    ("Ville-sur-Jarnioux", "69"),
]

lyon_metropole = [
    ("Albigny-sur-Saône", "69"),
    ("Bron", "69"),
    ("Cailloux-sur-Fontaines", "69"),
    ("Caluire-et-Cuire", "69"),
    ("Champagne-au-Mont-d'Or", "69"),
    ("Charbonnières-les-Bains", "69"),
    ("Charly", "69"),
    ("Chassieu", "69"),
    ("Collonges-au-Mont-d'Or", "69"),
    ("Corbas", "69"),
    ("Couzon-au-Mont-d'Or", "69"),
    ("Craponne", "69"),
    ("Curis-au-Mont-d'Or", "69"),
    ("Dardilly", "69"),
    ("Décines-Charpieu", "69"),
    ("Écully", "69"),
    ("Feyzin", "69"),
    ("Fleurieu-sur-Saône", "69"),
    ("Fontaines-Saint-Martin", "69"),
    ("Fontaines-sur-Saône", "69"),
    ("Francheville", "69"),
    ("Genay", "69"),
    ("Givors", "69"),
    ("Grigny-sur-Rhône", "69"),
    ("Irigny", "69"),
    ("Jonage", "69"),
    ("La Mulatière", "69"),
    ("La Tour-de-Salvagny", "69"),
    ("Limonest", "69"),
    ("Lissieu", "69"),
    ("Lyon", "69"),
    ("Marcy-l'Étoile", "69"),
    ("Meyzieu", "69"),
    ("Mions", "69"),
    ("Montanay", "69"),
    ("Neuville-sur-Saône", "69"),
    ("Oullins-Pierre-Bénite", "69"),
    ("Poleymieux-au-Mont-d'Or", "69"),
    ("Quincieux", "69"),
    ("Rillieux-la-Pape", "69"),
    ("Rochetaillée-sur-Saône", "69"),
    ("Saint-Cyr-au-Mont-d'Or", "69"),
    ("Saint-Didier-au-Mont-d'Or", "69"),
    ("Saint-Fons", "69"),
    ("Saint-Genis-Laval", "69"),
    ("Saint-Genis-les-Ollières", "69"),
    ("Saint-Germain-au-Mont-d'Or", "69"),
    ("Saint-Priest", "69"),
    ("Saint-Romain-au-Mont-d'Or", "69"),
    ("Sainte-Foy-lès-Lyon", "69"),
    ("Sathonay-Camp", "69"),
    ("Sathonay-Village", "69"),
    ("Solaize", "69"),
    ("Tassin-la-Demi-Lune", "69"),
    ("Vaulx-en-Velin", "69"),
    ("Vénissieux", "69"),
    ("Vernaison", "69"),
    ("Villeurbanne", "69"),
]

study_area_reference = pd.DataFrame(
    villefranche_agglo + lyon_metropole,
    columns=[
        "municipality",
        "department_code",
    ],
)

In [ ]:
# ============================================================================
# Build case-study area
# ============================================================================

study_area = municipality_geo.merge(
    study_area_reference,
    on=[
        "municipality",
        "department_code",
    ],
    how="inner",
)

print(
    f"Expected municipalities: "
    f"{len(study_area_reference)}"
)

print(
    f"Matched municipalities: "
    f"{len(study_area)}"
)

In [ ]:
# ============================================================================
# Inspect case-study municipalities
# ============================================================================


study_area[
    [
        "insee_code",
        "municipality",
        "department_code",
    ]
].sort_values(
    [
        "department_code",
        "municipality",
    ]
)

In [ ]:
# ============================================================================
# Build case-study geometry
# ============================================================================

study_area_geometry = (
    study_area[
        ["geometry"]
    ]
    .dissolve()
)

study_area_geometry

In [ ]:
# ============================================================================
# Inspect case-study geometry
# ============================================================================

print(f"CRS: {study_area_geometry.crs}")
print(
    f"Geometry type: "
    f"{study_area_geometry.geometry.iloc[0].geom_type}"
)

print(
    f"Bounds: "
    f"{study_area_geometry.total_bounds}"
)

In [ ]:
# ============================================================================
# Plot HERMES case-study area
# ============================================================================

fig, ax = plt.subplots(
    figsize=(8, 10)
)

study_area.boundary.plot(
    ax=ax,
    linewidth=0.6,
)

study_area_geometry.boundary.plot(
    ax=ax,
    linewidth=1.5,
)

ax.set_title(
    "HERMES case-study area\n"
    "Villefranche Beaujolais Saône + Métropole de Lyon"
)

ax.set_axis_off()

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "hermes_case_study_area.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================================
# Project case-study area to Lambert-93
# ============================================================================

study_area_l93 = study_area.to_crs(
    "EPSG:2154"
)

study_area_geometry_l93 = (
    study_area_l93[["geometry"]]
    .dissolve()
)

print(
    f"CRS: {study_area_l93.crs}"
)

print(
    f"Bounds: {study_area_geometry_l93.total_bounds}"
)

In [ ]:
# ============================================================================
# Compute case-study area
# ============================================================================

study_area_km2 = (
    study_area_geometry_l93.geometry.area.iloc[0]
    / 1_000_000
)

print(
    f"Case-study area: "
    f"{study_area_km2:,.1f} km²"
)

In [ ]:
# ============================================================================
# Identify RGE ALTI tiles covering the case-study area
# ============================================================================

study_geometry = (
    study_area_geometry_l93
    .geometry
    .iloc[0]
)

selected_tiles = build_tile_grid(
    study_geometry
)

mnt_tiles = build_tile_grid(
    study_geometry,
    buffer=20,
)

print(
    f"Study-area RGE ALTI tiles: "
    f"{len(selected_tiles):,}"
)

print(
    f"MNT tiles with buffer: "
    f"{len(mnt_tiles):,}"
)

print(
    f"Additional context tiles: "
    f"{len(mnt_tiles) - len(selected_tiles):,}"
)

In [ ]:
# ============================================================================
# Plot selected DEM tiles
# ============================================================================

fig, ax = plt.subplots(
    figsize=(8, 10)
)

selected_tiles.boundary.plot(
    ax=ax,
    linewidth=0.2,
)

study_area_l93.boundary.plot(
    ax=ax,
    linewidth=0.8,
)

ax.set_title(
    "RGE ALTI 1 m tile coverage\n"
    "HERMES case-study area"
)

ax.set_axis_off()

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "rge_alti_tile_coverage.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================================
# Download RGE ALTI data
# ============================================================================

download_dataset("elevation_rhone_raw")
download_dataset("elevation_ain_raw")

In [ ]:
# ============================================================================
# Prepare case-study RGE ALTI tiles
# ============================================================================

context_tile_ids = (
    set(mnt_tiles["ign_tile_id"])
    - set(selected_tiles["ign_tile_id"])
)

mnt_files = prepare_case_study_mnt(
    selected_tiles=mnt_tiles,
    rhone_archive=get_dataset(
        "elevation_rhone_raw"
    ).download_path,
    ain_archive=get_dataset(
        "elevation_ain_raw"
    ).download_path,
    destination=RGE_ALTI_MNT_DIR,
    optional_tile_ids=context_tile_ids,
)

In [ ]:
# ============================================================================
# Inspect one RGE ALTI MNT tile
# ============================================================================

sample_mnt_path = mnt_files[0]

with rasterio.open(sample_mnt_path) as src:

    print(f"File: {sample_mnt_path.name}")
    print(f"CRS: {src.crs}")
    print(f"Bounds: {src.bounds}")
    print(f"Resolution: {src.res}")
    print(f"Shape: {src.height} × {src.width}")
    print(f"NoData: {src.nodata}")

    elevation = src.read(
        1,
        masked=True,
    )

print()
print(
    f"Minimum elevation: {elevation.min():.1f} m"
)
print(
    f"Mean elevation: {elevation.mean():.1f} m"
)
print(
    f"Maximum elevation: {elevation.max():.1f} m"
)

In [ ]:
# ============================================================================
# Plot one RGE ALTI MNT tile
# ============================================================================

fig, ax = plt.subplots(
    figsize=(8, 7)
)

image = ax.imshow(
    elevation,
    cmap="terrain",
)

fig.colorbar(
    image,
    ax=ax,
    label="Elevation (m)",
)

ax.set_title(
    f"RGE ALTI 1 m — {sample_mnt_path.stem}"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Compare slope across spatial resolutions
# ============================================================================

slope_comparison = compare_slope_resolutions(
    sample_mnt_path
)

display(
    slope_comparison.round(2)
)

In [ ]:
# ============================================================================
# Compare slope resolutions across representative municipalities
# ============================================================================

terrain_samples = [
    "Lyon",
    "Villefranche-sur-Saône",
    "Saint-Cyr-au-Mont-d'Or",
    "Poleymieux-au-Mont-d'Or",
]

comparison_rows = []

for municipality_name in terrain_samples:

    geometry = (
        study_area_l93.loc[
            study_area_l93["municipality"]
            == municipality_name,
            "geometry",
        ]
        .iloc[0]
    )

    statistics = (
        compare_municipality_slope_resolutions(
            geometry,
            mnt_files,
        )
        .reset_index()
    )

    statistics.insert(
        0,
        "municipality",
        municipality_name,
    )

    comparison_rows.append(
        statistics
    )

terrain_resolution_comparison = pd.concat(
    comparison_rows,
    ignore_index=True,
)

display(
    terrain_resolution_comparison.round(2)
)

In [ ]:
# ============================================================================
# Build HERMES case-study DEM
# ============================================================================

case_study_geometry = (
    study_area_geometry_l93
    .geometry
    .union_all()
)

dem_path = build_case_study_dem(
    mnt_files=mnt_files,
    geometry=case_study_geometry,
    output_path=CASE_STUDY_DEM_PATH,
    resolution=10,
    buffer=20,
)

print(
    f"HERMES DEM created: {dem_path}"
)

In [ ]:
# ============================================================================
# Validate HERMES case-study DEM
# ============================================================================

with rasterio.open(dem_path) as src:

    dem = src.read(
        1,
        masked=True,
    )

    print(f"CRS: {src.crs}")
    print(f"Resolution: {src.res}")
    print(
        f"Shape: {src.height:,} × {src.width:,}"
    )
    print(
        f"Valid cells: {dem.count():,}"
    )
    print(
        f"Minimum elevation: {dem.min():.1f} m"
    )
    print(
        f"Mean elevation: {dem.mean():.1f} m"
    )
    print(
        f"Maximum elevation: {dem.max():.1f} m"
    )

In [ ]:
# ============================================================================
# Validate DEM spatial coverage
# ============================================================================

geometry_area_km2 = (
    case_study_geometry.area
    / 1_000_000
)

with rasterio.open(dem_path) as src:

    dem = src.read(
        1,
        masked=True,
    )

    pixel_area_m2 = (
        abs(src.res[0])
        * abs(src.res[1])
    )

    raster_area_km2 = (
        dem.count()
        * pixel_area_m2
        / 1_000_000
    )

coverage_pct = (
    raster_area_km2
    / geometry_area_km2
    * 100
)

print(
    f"Study-area geometry: "
    f"{geometry_area_km2:.2f} km²"
)

print(
    f"Valid DEM coverage: "
    f"{raster_area_km2:.2f} km²"
)

print(
    f"Coverage: "
    f"{coverage_pct:.2f} %"
)

In [ ]:
# ============================================================================
# Diagnose DEM coverage
# ============================================================================

with rasterio.open(dem_path) as src:

    dem = src.read(
        1,
        masked=True,
    )

    valid_mask = (
        ~np.ma.getmaskarray(dem)
    ).astype(
        np.uint8
    )

    raster_shapes = shapes(
        valid_mask,
        mask=valid_mask.astype(bool),
        transform=src.transform,
    )

    valid_geometries = [
        shape(geometry)
        for geometry, value in raster_shapes
        if value == 1
    ]

dem_coverage = gpd.GeoSeries(
    valid_geometries,
    crs="EPSG:2154",
).union_all()

missing_geometry = (
    case_study_geometry
    .difference(dem_coverage)
)

print(
    f"Missing area: "
    f"{missing_geometry.area / 1_000_000:.2f} km²"
)

print(
    f"Missing geometry type: "
    f"{missing_geometry.geom_type}"
)

In [ ]:
# ============================================================================
# Visualize missing DEM coverage
# ============================================================================

fig, ax = plt.subplots(
    figsize=(10, 10)
)

study_area_l93.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.5,
)

gpd.GeoSeries(
    missing_geometry,
    crs="EPSG:2154",
).plot(
    ax=ax,
    alpha=0.8,
)

ax.set_title(
    "Missing RGE ALTI coverage"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Diagnose source mosaic coverage
# ============================================================================

from rasterio.enums import Resampling
from rasterio.merge import merge

sources = [
    rasterio.open(path)
    for path in mnt_files
]

try:
    mosaic, mosaic_transform = merge(
        sources,
        res=10,
        resampling=Resampling.average,
        nodata=np.nan,
        dtype="float32",
    )

finally:
    for source in sources:
        source.close()

source_dem = mosaic[0]

print(
    f"Mosaic shape: "
    f"{source_dem.shape[0]:,} × "
    f"{source_dem.shape[1]:,}"
)

print(
    f"NaN cells in complete mosaic: "
    f"{np.isnan(source_dem).sum():,}"
)

print(
    f"Valid cells in complete mosaic: "
    f"{np.isfinite(source_dem).sum():,}"
)

In [ ]:
# ============================================================================
# Compare source coverage with case-study geometry
# ============================================================================

from rasterio.features import geometry_mask
from shapely.geometry import mapping

inside = geometry_mask(
    [mapping(case_study_geometry)],
    out_shape=source_dem.shape,
    transform=mosaic_transform,
    invert=True,
)

inside_cells = inside.sum()

valid_inside = (
    inside
    & np.isfinite(source_dem)
).sum()

missing_inside = (
    inside
    & ~np.isfinite(source_dem)
).sum()

print(
    f"Cells inside study area: "
    f"{inside_cells:,}"
)

print(
    f"Valid source cells inside study area: "
    f"{valid_inside:,}"
)

print(
    f"Missing source cells inside study area: "
    f"{missing_inside:,}"
)

print(
    f"Source coverage inside study area: "
    f"{valid_inside / inside_cells * 100:.2f} %"
)

## Build terrain slope

Terrain slope is one of the main topographic constraints affecting cycling feasibility.

The validated 10 m HERMES digital elevation model is used to derive a slope raster at the same spatial resolution.

Slope is expressed as percent grade and will provide the basis for subsequent terrain features used in the cycling feasibility model.

In [ ]:
# ============================================================================
# Build terrain slope raster
# ============================================================================

slope_path = build_slope_raster(
    dem_path=CASE_STUDY_DEM_PATH,
    output_path=CASE_STUDY_SLOPE_PATH,
)

print(
    f"Slope raster: {slope_path}"
)

## Inspect terrain slope raster

The generated slope raster is inspected to verify its spatial properties, valid-data coverage and slope-value distribution.

The slope raster should preserve the CRS, extent and 10 m resolution of the case-study DEM. Summary statistics are also examined to identify potentially implausible terrain values before deriving modeling features.

In [ ]:
# ============================================================================
# Inspect terrain slope raster
# ============================================================================

with rasterio.open(CASE_STUDY_SLOPE_PATH) as src:

    slope = src.read(
        1,
        masked=True,
    )

    print(
        f"CRS: {src.crs}"
    )
    print(
        f"Resolution: {src.res}"
    )
    print(
        f"Shape: {src.width:,} × {src.height:,}"
    )
    print(
        f"Bounds: {src.bounds}"
    )
    print(
        f"NoData: {src.nodata}"
    )

    valid_slope = slope.compressed()

print()
print(
    f"Valid cells: {valid_slope.size:,}"
)
print(
    f"Minimum slope: {valid_slope.min():.2f} %"
)
print(
    f"Mean slope: {valid_slope.mean():.2f} %"
)
print(
    f"Median slope: {np.median(valid_slope):.2f} %"
)
print(
    f"P90 slope: {np.percentile(valid_slope, 90):.2f} %"
)
print(
    f"P95 slope: {np.percentile(valid_slope, 95):.2f} %"
)
print(
    f"P99 slope: {np.percentile(valid_slope, 99):.2f} %"
)
print(
    f"Maximum slope: {valid_slope.max():.2f} %"
)

In [ ]:
# ============================================================================
# Plot slope distribution
# ============================================================================

fig, ax = plt.subplots(
    figsize=(10, 5)
)

ax.hist(
    valid_slope,
    bins=100,
)

ax.set(
    title="Distribution of terrain slope",
    xlabel="Slope (%)",
    ylabel="Number of cells",
)

plt.tight_layout()
plt.show()

### Inspect extreme slope values

The slope distribution contains a small number of extreme values.

Although slopes above 100% are mathematically possible, very high values may also result from local elevation discontinuities or raster boundary effects.

Their frequency and spatial distribution are therefore examined before any terrain filtering or feature engineering is applied.

In [ ]:
# ============================================================================
# Inspect extreme slope values
# ============================================================================

thresholds = (
    50,
    75,
    100,
    150,
    200,
)

for threshold in thresholds:

    count = np.sum(
        valid_slope > threshold
    )

    share = (
        count
        / valid_slope.size
        * 100
    )

    print(
        f"Slope > {threshold:3d} %: "
        f"{count:>8,} cells "
        f"({share:.4f} %)"
    )

In [ ]:
# ============================================================================
# Plot slope distribution below P99
# ============================================================================

p99 = np.percentile(
    valid_slope,
    99,
)

fig, ax = plt.subplots(
    figsize=(10, 5)
)

ax.hist(
    valid_slope[
        valid_slope <= p99
    ],
    bins=100,
)

ax.set(
    title="Distribution of terrain slope below P99",
    xlabel="Slope (%)",
    ylabel="Number of cells",
)

ax.axvline(
    np.median(valid_slope),
    linestyle="--",
    label="Median",
)

ax.axvline(
    np.percentile(valid_slope, 90),
    linestyle="--",
    label="P90",
)

ax.legend()

plt.tight_layout()
plt.show()

### Map extreme slope values

Extreme slope values represent only a very small fraction of the study area.

Their spatial distribution is examined to determine whether they correspond to genuine steep terrain or to raster artefacts associated with boundaries, NoData cells or local elevation discontinuities.

In [ ]:
# ============================================================================
# Map extreme slope values
# ============================================================================

with rasterio.open(CASE_STUDY_SLOPE_PATH) as src:

    slope = src.read(
        1,
        masked=True,
    )

    extent = (
        src.bounds.left,
        src.bounds.right,
        src.bounds.bottom,
        src.bounds.top,
    )

extreme_slope = np.ma.masked_where(
    slope <= 100,
    slope,
)

fig, ax = plt.subplots(
    figsize=(10, 12)
)

study_area_l93.boundary.plot(
    ax=ax,
    linewidth=0.5,
)

image = ax.imshow(
    extreme_slope,
    extent=extent,
    origin="upper",
)

fig.colorbar(
    image,
    ax=ax,
    label="Slope (%)",
    shrink=0.7,
)

ax.set(
    title="Terrain cells with slope > 100%",
    xlabel="Lambert-93 easting (m)",
    ylabel="Lambert-93 northing (m)",
)

plt.tight_layout()
plt.show()

### Check extreme slopes near NoData

Extreme slopes do not appear to follow the outer study-area boundary.

A final diagnostic tests whether these values occur disproportionately close to NoData cells, which could indicate local gradient artefacts around gaps in the elevation raster.

In [ ]:
# ============================================================================
# Check extreme slopes near NoData
# ============================================================================

with rasterio.open(CASE_STUDY_DEM_PATH) as src:

    elevation = src.read(
        1,
        masked=True,
    )

dem_invalid = np.ma.getmaskarray(
    elevation
)

extreme = (
    (~np.ma.getmaskarray(slope))
    & (slope.data > 100)
)

# Cells directly adjacent to NoData:
# horizontal, vertical or diagonal neighbours.

near_nodata = np.zeros_like(
    dem_invalid,
    dtype=bool,
)

for dy in (-1, 0, 1):
    for dx in (-1, 0, 1):

        if dx == 0 and dy == 0:
            continue

        shifted = np.roll(
            dem_invalid,
            shift=(dy, dx),
            axis=(0, 1),
        )

        near_nodata |= shifted

extreme_count = extreme.sum()

extreme_near_nodata = (
    extreme
    & near_nodata
).sum()

print(
    f"Extreme slope cells (>100%): "
    f"{extreme_count:,}"
)

print(
    f"Extreme cells adjacent to NoData: "
    f"{extreme_near_nodata:,}"
)

print(
    f"Share adjacent to NoData: "
    f"{extreme_near_nodata / extreme_count * 100:.2f} %"
)

### Slope raster validation

The 10 m slope raster preserves the spatial properties of the HERMES
case-study DEM and exhibits a plausible right-skewed terrain distribution.

Extreme slopes are rare: only 0.036% of valid cells exceed 100% grade.
Their spatial distribution does not follow the study-area boundary, and only 0.04% of these extreme cells are adjacent to NoData areas.

The extreme values therefore do not appear to result from boundary or NoData gradient artefacts. The slope raster is retained without arbitrary clipping and is considered suitable for subsequent terrain feature engineering.

## Prepare network terrain features

Terrain characteristics must ultimately be associated with the transport network rather than analysed only as raster properties.

For cycling applications, elevation change is directional: the same road segment represents an ascent in one direction and a descent in the opposite direction.

The HERMES digital elevation model will therefore be combined with the network geometry to derive segment-level elevation and slope features.

In [ ]:
# ============================================================================
# Inspect available GeoDataFrames
# ============================================================================

for name, obj in globals().items():

    if isinstance(obj, gpd.GeoDataFrame):

        print(
            f"{name}: "
            f"{len(obj):,} rows | "
            f"CRS={obj.crs} | "
            f"geometry={obj.geometry.geom_type.unique()}"
        )

In [ ]:
study_area_l93.columns.tolist()

In [ ]:
# ============================================================================
# Compute municipality terrain features
# ============================================================================

municipality_terrain = compute_municipality_terrain_features(
    municipalities=study_area_l93,
    dem_path=CASE_STUDY_DEM_PATH,
    slope_path=CASE_STUDY_SLOPE_PATH,
)

print(
    f"Municipalities: {len(municipality_terrain):,}"
)

municipality_terrain.head()

In [ ]:
# ============================================================================
# Inspect selected municipality terrain profiles
# ============================================================================

terrain_check = (
    municipality_terrain[
        municipality_terrain["municipality"].isin(
            [
                "Lyon",
                "Villefranche-sur-Saône",
                "Saint-Cyr-au-Mont-d'Or",
                "Poleymieux-au-Mont-d'Or",
            ]
        )
    ]
    .sort_values(
        "slope_mean_pct"
    )
    .round(2)
)

terrain_check